# 10 — Validate Against DFT Reference Data

Validates ML predictions against DFT calculations for selected configurations on and near the convex hull.

## Prerequisites / Input files
- `Fe-Mo/FullyCuratedParsedBriefSummary.pkl`
- `Fe-Mo/data/Validation/` — DFT validation data (included in repo)
- `Fe-Mo/results/voting_regressor_*.pkl` — ensemble models

## Outputs
- Validation metrics and figures in `Fe-Mo/graphs/`



In [ ]:
from Tools.DatasetTools.DatasetOperator import Dataset
from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.brief_summary_parser import StructSummaryParser
from Tools.DatasetTools import EVCurvesTools as EVtools
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [ ]:
from importlib.machinery import SourceFileLoader

In [ ]:
fullvalidationBS = pd.read_pickle('validation_data/FullyCuratedParsedBriefSummary.pkl')

In [ ]:
remove_phases_query='Phase != "bcc" and Phase != "fcc" and Phase !="hcp"',

In [ ]:
BS = pd.read_pickle('Fe-Mo/FullyCuratedParsedBriefSummary.pkl')

In [ ]:
plt.rc('text', usetex=False)

In [ ]:
BS['E0'].plot.hist()

In [ ]:
validation_samples = fullvalidationBS.index.difference(BS.index)

In [ ]:
validation_samples = validation_samples[~validation_samples.str.contains('.FM|sigma')]

In [ ]:
len(validation_samples)

In [ ]:
validationBS = fullvalidationBS.loc[validation_samples]

In [ ]:
plt.hist(validationBS['E0'])

TODO:
[ ] calculate formation energies for new calculations

In [ ]:
validationBS = validationBS[~validationBS.index.str.contains('sigma.[DU]')]

In [ ]:
validationBS

# Check EVcurves

In [ ]:
validation_dataset='validation_data'

In [ ]:
fittedcurvesloc = os.path.join(validation_dataset, 'evcurvesfitted.json')
evcurvesloc = os.path.join(validation_dataset,'evcurves.json' )
goodnessloc = os.path.join(validation_dataset, 'goodness.json')
force = False

In [ ]:
from Tools.DatasetTools import EVCurvesTools as EVtools

In [ ]:
#EVtools = SourceFileLoader('EVtools', 'Tools/DatasetTools/EVCurvesTools.py').load_module()

In [ ]:
test_indexes = validationBS.index[validationBS.index.str.contains('Mo_sv.*P.NM')]

In [ ]:
validationBS.loc[test_indexes]

In [ ]:
import subprocess
from pathlib import Path

mount_root = Path(validation_dataset).resolve()
target_dirs = ["Fe_pv", "Fe_pv-Mo_sv", "Mo_sv"]
target_paths = [(mount_root / directory).resolve() for directory in target_dirs]

def _read_mounts():
    mounts = {}
    with open("/proc/mounts", "r", encoding="utf-8") as mount_file:
        for line in mount_file:
            parts = line.split()
            if len(parts) >= 3:
                mount_point = parts[1].replace("\\040", " ")
                fs_type = parts[2]
                mounts[mount_point] = fs_type
    return mounts

mounts = _read_mounts()
missing_fuse_mounts = [
    path for path in target_paths
    if mounts.get(str(path), "").startswith("fuse") is False
]

if missing_fuse_mounts:
    script_path = mount_root / "mount_data.sh"
    if not script_path.exists():
        raise FileNotFoundError(f"Missing mount script: {script_path}")
    print("Missing FUSE mounts detected. Running mount_data.sh...")
    subprocess.run(["bash", script_path.name], cwd=str(mount_root), check=True)
else:
    print("All target directories are already mounted with FUSE. Skipping mount_data.sh.")

In [ ]:
# NOTE: EV curve fitting requires raw DFT data (not included in this repository).
# Download raw data from NOMAD or contact the authors.
if os.path.exists(os.path.join(validation_dataset, 'Fe_pv', 'briefsummary.dat')):
    if not os.path.exists(fittedcurvesloc) or force:
        if not os.path.exists(evcurvesloc) or force:
            EV = EVtools.Evcurves(Indexes = validationBS.index, atoms=['Fe','Mo'], dataset = validation_dataset)#, search_str='**/volume_relaxed/**/volume-energy.dat')
            EV.load_evcurves( deltaks = validationBS['deltak'], encuts = validationBS['encut'])
            EVcurves = EV.evcurves
            EVcurves.to_json(evcurvesloc)
        else:
            EVcurves = pd.read_json(evcurvesloc, typ='series')
        goodness, fiteos, r2  = EVtools.get_goodness(EVcurves, r2tol = 1e-6)
        if goodness.map(lambda g: False in g.values()).all():
            goodness = EVtools.invert_goodness(goodness)
        Goodness = pd.Series(goodness)
        Fits = pd.Series(fiteos)
        R2 = pd.Series(r2)
        for index, data in EVcurves.items():
            for key, evcurve in data.items():
                if index in R2.keys():
                    if key in R2[index].keys():
                        data[key].update({'r2': R2[index][key], 'fit': Fits[index][key], 'IsGood': Goodness[index][key]})
        EVcurves.to_json(fittedcurvesloc)
    else:
        print('B')
        EVcurves = pd.read_json(fittedcurvesloc, typ='series')
        R2 = EVtools.get_key_for_curves(EVcurves, 'r2')
        Goodness = EVtools.get_key_for_curves(EVcurves, 'IsGood')
        Fits = EVtools.get_key_for_curves(EVcurves, 'fit')

In [ ]:
EVcurves

# Recover Predicted values

In [ ]:
validationBS.loc[test_indexes]['E0']

In [ ]:
for index, param_curve in EVcurves[test_indexes].items():
    break

In [ ]:
Fits[index]

In [ ]:
for params, curve in param_curve.items():
    break

In [ ]:
fig, ax = plt.subplots()
EVtools.plot_fitted_curve(curve['evcurve'], Fits[index][params], R2[index][params], ax = ax, fig=fig)
ax.plot([validationBS.loc[index]['V0']], [validationBS.loc[index]['E0']],'o')

In [ ]:
EVcurves_df = pd.DataFrame.from_dict(EVcurves.to_dict(), orient='index')

In [ ]:
empty_curves = EVcurves[EVcurves.map(lambda c: len(c)==0)].index

In [ ]:
EVcurves[empty_curves]

In [ ]:
complete_curves = EVcurves.index.difference(empty_curves)

Only take the validationBS with given EV curves

In [ ]:
validationBS = validationBS.loc[complete_curves]
EVcurves = EVcurves[complete_curves]
R2 = R2[complete_curves]
Fits = Fits[complete_curves]

# Get dataframes 

In [ ]:
R2_df = pd.DataFrame.from_dict(R2.to_dict(), orient='index')

In [ ]:
Fits_df = pd.DataFrame.from_dict(Fits.to_dict(), orient='index')

In [ ]:
Better_evcurves = {}

In [ ]:
for index, evcurves in EVcurves_df.iterrows():
    nonan_evcurves = evcurves.dropna()
    if len(nonan_evcurves) > 1:
        break
    nonan_evcurves[nonan_evcurves.index][0]['calc_param'] = nonan_evcurves.index[0]
    Better_evcurves[index] = nonan_evcurves[nonan_evcurves.index][0]

In [ ]:
Better_evcurves_df = pd.DataFrame.from_dict(Better_evcurves, orient='index')

In [ ]:
ev_fit_results_df = pd.DataFrame.from_dict(Better_evcurves_df.ev_fit_results.to_dict(), orient='index')

In [ ]:
fit_results = Better_evcurves_df.fit.map( lambda f : {name: val for name, val in zip(['E_murn', 'B_murn', 'Bdev_murn', 'V_murn'], f)})

In [ ]:
fit_results_df = pd.DataFrame.from_dict(fit_results.to_dict(), orient='index')

In [ ]:
ax = fit_results_df[Better_evcurves_df.IsGood]['B_murn'].plot.hist(bins=100)
ax.set_xlabel(xlabel=r'$B_0$ (GPa)')
Ngood = Better_evcurves_df.IsGood.sum()
ax.set_title(f'{Ngood} Good Samples')

In [ ]:
Better_evcurves_df.shape

In [ ]:
indexofgoodsamples = Better_evcurves_df.index[Better_evcurves_df.IsGood]

In [ ]:
indexofbadsamples = Better_evcurves_df.index.difference(indexofgoodsamples) #[~Better_evcurves_df.IsGood]

In [ ]:
indexofbadsamples

In [ ]:
Better_evcurves_df.index.difference(validationBS.index)

In [ ]:
ax = fit_results_df[Better_evcurves_df.IsGood]['B_murn'].plot.hist(bins=100)
ax.set_xlabel(xlabel=r'$B_0$ (GPa)')
Ngood = Better_evcurves_df.IsGood.sum()
ax.set_title(f'{Ngood} Good Samples')

In [ ]:
indexofgoodsamples = Better_evcurves_df.index[Better_evcurves_df.IsGood]

In [ ]:
indexofbadsamples = Better_evcurves_df.index.difference(indexofgoodsamples) #[~Better_evcurves_df.IsGood]

In [ ]:
Better_evcurves_df.index.difference(validationBS.index)

In [ ]:
hist, bins = np.histogram(1-Better_evcurves_df.r2[indexofgoodsamples], bins=100)
logbins = np.logspace(np.log10(bins[0]), np.log10(bins[-1]), len(bins))
fig, ax = plt.subplots()
ax.hist(1-Better_evcurves_df.r2[indexofgoodsamples], bins=logbins)
Better_evcurves_df.r2[indexofgoodsamples]
ax.set_xscale('log')
xlabel = ax.set_xlabel('$1-R^2$')
NGOOD = len(indexofgoodsamples)
ax.legend(title=f'{NGOOD} good samples')

# Differences between fits and available data for bad samples

In [ ]:
diff_fit_to_dataset = ((validationBS.B0[indexofbadsamples] - fit_results_df.B_murn[indexofbadsamples])/fit_results_df.B_murn[indexofbadsamples]).abs().to_frame().rename(columns={0: 'B0'})

In [ ]:
diff_fit_to_dataset['E0'] = ((validationBS.E0[indexofbadsamples] - fit_results_df.E_murn[indexofbadsamples])/fit_results_df.E_murn[indexofbadsamples]).abs().dropna()

In [ ]:
diff_fit_to_dataset['V0'] = ((validationBS.V0[indexofbadsamples] - fit_results_df.V_murn[indexofbadsamples])/fit_results_df.V_murn[indexofbadsamples]).abs().dropna()

In [ ]:
large_diff_E0 = diff_fit_to_dataset.query('E0 > 1e-5').index
fig, ax = plt.subplots()
hist, bins = np.histogram(diff_fit_to_dataset.E0, bins=100)
logbins = np.logspace(np.log10(bins[0]), np.log10(bins[-1]), len(bins))
loghist = plt.hist(diff_fit_to_dataset.E0, bins=logbins)
fig = plt.xscale('log')
fig = plt.yscale('log')
for index in large_diff_E0:
    x = diff_fit_to_dataset.E0[index]
    y = 1
    ax.annotate(index, (x, y), rotation='90')
plt.xlabel(r'$|(E_0 ^{fit} - E_0 ^{BS})/E_0^{BS}|$')

# Try to correct the bad fits by removing points 

In [ ]:
from importlib.machinery import SourceFileLoader

In [ ]:
len(validationBS)

In [ ]:
len(indexofgoodsamples)

In [ ]:
len(indexofbadsamples)

In [ ]:
GoodBS = validationBS.loc[indexofgoodsamples]

In [ ]:
from importlib.machinery import SourceFileLoader
find_the_good_curve_inside = SourceFileLoader('find_the_good_curve_inside','Tools/DatasetTools/EVCurvesTools.py').load_module().find_the_good_curve_inside
is_common_sense_evcurve = SourceFileLoader('is_common_sense_evcurve','Tools/DatasetTools/EVCurvesTools.py').load_module().is_common_sense_evcurve
ev_per_angstrom3_to_GPA = 160.21

doexit = False

fixedevcurves = pd.Series([], name='FixedEVcurves')
fixedr2 = pd.Series([], name='FixedR2')
fixedfit = pd.Series([], name='Fixedfit')
tol = 1e-6
now_is_good = []
common_sense_evcurve = []
progress = tqdm(EVcurves[indexofbadsamples].items(), total = len(indexofbadsamples))
for index, paramcurve in progress:
    if index in fixedevcurves.keys():
        continue
    progress.set_description('index')
    if index in GoodBS.index:
        continue
    for paramspec, curvedata in paramcurve.items():
        r2, params_onreduced, reducedv, reducede = find_the_good_curve_inside(curvedata, tol=tol, reset_guess_params = True)
        common_sense_evcurve.append( is_common_sense_evcurve(reducedv, reducede, params_onreduced, unitsofb0='GPa'))
        now_is_good.append( (r2>1-tol) & common_sense_evcurve[-1] )
        progress.set_postfix_str(f'{index}, 1-r2 = {1-r2:.2e}, B0={params_onreduced[1]}, now is good = {now_is_good[-1]}')
        if params_onreduced[1] < 0 and now_is_good:
            raise ValueError('B0 is negative on '+index)
        fixedevcurves[index] = {
            paramspec: {
                'evcurve' :{ 'V' : reducedv , 'E': reducede },
                'ev_fit_results' :  {'E_murn': params_onreduced[0], 'V_murn' : params_onreduced[-1], 'B_murn': params_onreduced[1], 'Bdev_murn' : params_onreduced[2]},
                'r2' : r2,
                'IsGood' : now_is_good[-1],
                'fit': params_onreduced,
                'calc_param' : paramspec
            }
        }

In [ ]:
fixedevcurves

In [ ]:
fixedevcurves_df = pd.DataFrame.from_dict(fixedevcurves.to_dict(), orient='index')

In [ ]:
fixedevcurves_df.shape

In [ ]:
Better_fixedevcurves = {}

In [ ]:
for index, evcurves in fixedevcurves_df.iterrows():
    nonan_evcurves = evcurves.dropna()
    if len(nonan_evcurves) > 1:
        break
    nonan_evcurves[nonan_evcurves.index][0]['calc_param'] = nonan_evcurves.index[0]
    Better_fixedevcurves[index] = nonan_evcurves[nonan_evcurves.index][0]

In [ ]:
Better_fixedevcurves_df = pd.DataFrame.from_dict(Better_fixedevcurves, orient='index')

In [ ]:
fixedR2 = pd.Series([])
for index, curve in fixedevcurves.items():
    for paramspec, curvedata in curve.items():
        fixedR2[index] = {paramspec: curvedata['r2']}

In [ ]:
fixedFits = pd.Series([])
for index, curve in fixedevcurves.items():
    for paramspec, curvedata in curve.items():
        fixedFits[index] = {paramspec: curvedata['fit']}

In [ ]:
fixed_ev_fit_results = {} #pd.DataFrame([])
for index, curve in fixedevcurves.items():
    for paramspec, curvedata in curve.items():
        fixed_ev_fit_results[index] = pd.Series(curvedata['ev_fit_results'])
fixed_ev_fit_results_df = pd.DataFrame.from_dict(fixed_ev_fit_results, orient='index')

In [ ]:
indexoffixedgoodsamples = Better_fixedevcurves_df.query('IsGood == True').index

In [ ]:
indexoffixedbadsamples = Better_fixedevcurves_df.index.difference(indexoffixedgoodsamples)

In [ ]:
indexoffixedbadsamples

In [ ]:
len(indexoffixedgoodsamples) + len(indexofgoodsamples)#+len(indexoffixedbadsamples)

In [ ]:
finalindexofsamples = indexofgoodsamples.append(indexoffixedgoodsamples)

In [ ]:
finalindexofsamples

In [ ]:
fixedValidationBS = validationBS.loc[finalindexofsamples]

# Fixed quantities

In [ ]:
thebins = np.linspace(160, 300, 101)
ax = ev_fit_results_df.B_murn[indexofgoodsamples].plot.hist(bins=thebins)
fixed_ev_fit_results_df.B_murn[indexoffixedgoodsamples].plot.hist(bins=thebins, ax = ax, alpha=0.5, )
fixed_ev_fit_results_df.B_murn[indexoffixedgoodsamples].plot.hist(bins=thebins, ax=ax)
ax.legend([f'{len(indexoffixedgoodsamples)} fixed good samples', f'{len(indexofgoodsamples)} good samples', f'{len(indexoffixedbadsamples)} fixed bad samples'])

In [ ]:
thebins = np.linspace(-11, -9, 101)
ax = ev_fit_results_df.E_murn[indexofgoodsamples].plot.hist(bins=thebins)
fixed_ev_fit_results_df.E_murn[indexoffixedgoodsamples].plot.hist(bins=thebins, ax = ax, alpha=0.5, )
fixed_ev_fit_results_df.E_murn[indexoffixedgoodsamples].plot.hist(bins=thebins, ax=ax)
ax.legend([f'{len(indexoffixedgoodsamples)} fixed good samples', f'{len(indexofgoodsamples)} good samples', f'{len(indexoffixedbadsamples)} fixed bad samples'])

# redefine the new good BS

but now I need to correct the E0, B0 and V0 to the results of the fix !

In [ ]:
fixedValidationBS['E0'].dropna()

In [ ]:
fixedValidationBS.loc[indexoffixedgoodsamples,'E0'] = fixed_ev_fit_results_df['E_murn'][indexoffixedgoodsamples]
fixedValidationBS.loc[indexoffixedgoodsamples,'V0'] = fixed_ev_fit_results_df['V_murn'][indexoffixedgoodsamples]
fixedValidationBS.loc[indexoffixedgoodsamples,'B0'] = fixed_ev_fit_results_df['B_murn'][indexoffixedgoodsamples]
fixedValidationBS.dropna().describe()

In [ ]:
fixedValidationBS.shape

In [ ]:
fixedValidationBS['E0']

In [ ]:
import re

In [ ]:
def get_phase_from_index(theindex):
    phaseconfig = theindex.split('.')[1]
    phase = phaseconfig.split('-')[0]
    return phase

In [ ]:
fixedValidationBS['Phase'] = fixedValidationBS.index.map(get_phase_from_index)

# New data for delta using AMS tools

In [ ]:
from ase import Atoms
import pandas as pd

In [ ]:
import sys
import numpy.core as np_core
import numpy.core.numeric as np_numeric

# Compatibility shim for pickles written with NumPy internals under 'numpy._core'
sys.modules.setdefault('numpy._core', np_core)
sys.modules.setdefault('numpy._core.numeric', np_numeric)

_bs_w_r_path = 'Fe-Mo/inchulldft/BS_w_R.pkl.gzip'
if os.path.exists(_bs_w_r_path):
    ValidationBS_delta = pd.read_pickle(_bs_w_r_path, compression='gzip')
else:
    ValidationBS_delta = None
    print('Fe-Mo/inchulldft/BS_w_R.pkl.gzip not found — cells using ValidationBS_delta will be skipped.')

In [ ]:
if ValidationBS_delta is not None:
    ValidationBS_delta['ase_atoms'] = ValidationBS_delta['relax_optimized_structure'].map(lambda d: Atoms.fromdict(d))

In [ ]:
if ValidationBS_delta is not None:
    num_atoms_delta = pd.DataFrame.from_dict(ValidationBS_delta['ase_atoms'].map(lambda d: {s: n for s, n in d.symbols.formula.count().items()}).to_dict(), orient='index').fillna(0)

In [ ]:
if ValidationBS_delta is not None:
    ValidationBS_delta = pd.concat([ValidationBS_delta, num_atoms_delta], axis=1)

In [ ]:
def parse_name(thename):
    splittedname = thename.split('/')
    sample_id = splittedname[-3]
    sample_info = sample_id.split('.')
    return {'Mag': sample_info[-1], '' : sample_info[1], 'Phase' : sample_info[1].split('-')[0], 'sample_id': sample_id}

In [ ]:
if ValidationBS_delta is not None:
    IDS = pd.DataFrame.from_dict(ValidationBS_delta.name.map(parse_name).to_dict(), orient='index')

In [ ]:
if ValidationBS_delta is not None:
    ValidationBS_delta = pd.concat([IDS, ValidationBS_delta], axis=1)

In [ ]:
if ValidationBS_delta is not None:
    ValidationBS_delta.set_index('sample_id', inplace=True)

In [ ]:
if ValidationBS_delta is not None:
    ValidationBS_delta.columns

In [ ]:
if ValidationBS_delta is not None:
    ValidationBS_delta.rename(columns={'murnaghan_equilibrium_bulk_modulus': 'B0', 'Fe': 'num_atom_A', 'Mo': 'num_atom_B', 'relax_n_atom': 'num_atoms'}, inplace=True)

In [ ]:
if ValidationBS_delta is not None:
    ValidationBS_delta.columns

In [ ]:
if ValidationBS_delta is not None:
    ValidationBS_delta['E0'] = ValidationBS_delta['murnaghan_equilibrium_energy']/ValidationBS_delta['num_atoms']
    ValidationBS_delta['V0'] = ValidationBS_delta['murnaghan_equilibrium_volume']/ValidationBS_delta['num_atoms']

In [ ]:
if ValidationBS_delta is not None:
    ValidationBS_delta.head()['E0']

In [ ]:
def get_n_atom(natoms):
    if natoms > 0:
        return 1
    return 0

In [ ]:
if ValidationBS_delta is not None:
    #ValidationBS_delta['atom_A'] =
    ValidationBS_delta['atom_A'] = ValidationBS_delta['num_atom_A'].map(lambda a: get_n_atom(a)*'Fe_pv')

In [ ]:
if ValidationBS_delta is not None:
    #ValidationBS_delta['atom_A'] =
    ValidationBS_delta['atom_B'] = ValidationBS_delta['num_atom_B'].map(lambda a: get_n_atom(a)*'Mo_sv')

In [ ]:
fixedValidationBS.head()

In [ ]:
if ValidationBS_delta is not None:
    FullValidationBS = pd.concat([ValidationBS_delta, fixedValidationBS], axis=0)
else:
    FullValidationBS = fixedValidationBS.copy()

In [ ]:
FullValidationBS

# Getting Formation Energies

In [ ]:
from Tools.DatasetTools import ValidationSetNormalization as nv

In [ ]:
ground_state_energies ={ ('Fe_pv','NM'): -8.18407186323532,
                       ('Mo_sv', 'NM'): -10.932821158319513,
                      }
# taken from ground_state_energies in notebook 03

In [ ]:
normalizer = nv.briefsummary_normalizer(
    ground_states=ground_state_energies, init_bs=FullValidationBS, 
    atomA='Fe', atomB = 'Mo'
)

In [ ]:
FullValidationBS['EF'] = normalizer.get_formation_energies()

In [ ]:
FullValidationBS[[f'x_{normalizer.atomA}',f'x_{normalizer.atomB}' ]] = normalizer.get_atom_composition()

In [ ]:
FullValidationBS.to_pickle('validation_data/FullyCuratedParsedBriefSummary.pkl')

# saving to pickle

In [ ]:
FullValidationBS.to_pickle('validation_data/FullyCuratedParsedBriefSummary.pkl')

# Just a peak into the predictions

In [ ]:
just_a_prediction = pd.read_csv('Fe-Mo/results/PREDICTION__R__0.7dprojections_0.5os_16.csv', index_col=0)

In [ ]:
plt.hist(just_a_prediction['EF_nmhcp__0.7dprojections_0.5os'])

In [ ]:
validation_R = FullValidationBS.query('index.str.contains("R")')

In [ ]:
validated_predictions = validation_R.index.intersection(just_a_prediction.index)

In [ ]:
validated_predictions.str.contains('AAAAAABAABB').sum()

In [ ]:
plt.scatter(
    just_a_prediction['EF_nmhcp__0.7dprojections_0.5os'].loc[validated_predictions], 
    FullValidationBS.loc[validated_predictions]['EF']
)

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
#mean_squared_error(
#    validationBS['EF'][validated_predictions], just_a_prediction['EF_nmhcp__0.7dprojections_0.5os'][validated_predictions],
#    squared=False
#)